# Pipeline v4 — Distill-and-Solve

**Architecture**
- Phase 1: Demand Planner → demand_plan (1 LLM call, cached)
- Phase 2: Chunk Distillation → KnowledgeStore (1 LLM call/chunk, parallel, cached)
- Phase 3: Python-first solvers + LLM synthesis
  - Needle fallback: raw-text keyword search on original chunk paragraphs

**Key features vs v3**
- 1 LLM call/chunk instead of 3
- Typed, queryable KnowledgeStore (not operation-based ledger)
- Demand plan guides extraction before reading any chunk
- Normalized numeric values → deterministic Python sorting/counting
- Events timeline (year_approx) for temporal queries
- Contradiction detection is fully algorithmic (set-diff on normalized values)

In [ ]:
import os

# ⚠️  Set your API key here — never hardcode in committed files
# Set COURSE_API_KEY and COURSE_API_BASE_URL in the environment before starting Jupyter.

print('API key set:', bool(os.environ.get('COURSE_API_KEY')))

In [ ]:
import sys
import logging

# Add pipeline_v4 to path
sys.path.insert(0, '..')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(name)s  %(message)s',
    datefmt='%H:%M:%S',
)
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('openai').setLevel(logging.WARNING)

print('Imports OK')

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DOCUMENT_PATH = r'nationalparks_europe.txt'
YAML_PATH     = r'dev_questions.yaml'

# ── Run options ───────────────────────────────────────────────────────────────
CONCURRENCY      = 4      # parallel distillation workers
USE_CACHE        = False   # 원래 True , resume interrupted runs without re-distilling
FORCE_REPLAN     = True  # 원래 False , set True to regenerate demand_plan from scratch
QUESTION_IDS     = None   # e.g. ['d01','d07'] to run a subset; None = all
LOAD_STORE       = None   # path to saved knowledge_store.json to skip Phase 2

In [ ]:
# Change working directory to pipeline_v4 so relative output paths resolve
import os
os.chdir(os.path.dirname(os.path.abspath('main_v4.ipynb')))
print('Working dir:', os.getcwd())

In [ ]:
from src_v4.question_classifier import classify_questions_batch

# 라벨 없이 질문만 넣기
test_qs = [
    {"id": "t1", "question": "How many parks are in France?"},
    {"id": "t2", "question": "Which park has the largest area?"},
    {"id": "t3", "question": "Which of the following is NOT mentioned: poaching, tourism, or drought?"},
    {"id": "t4", "question": "Find a contradiction in the book about a park's elevation."},
    {"id": "t5", "question": "What year was Swiss National Park founded?"},
    {"id": "t6", "question": "How does the book as a whole portray conservation?"},
    {"id": "t7", "question": "What do Norwegian and Icelandic parks have in common geologically?"},
]

results = classify_questions_batch(test_qs)
for qid, qtype in results.items():
    q = next(q["question"] for q in test_qs if q["id"] == qid)
    print(f"[{qtype:<18}] {q[:60]}")

In [ ]:
from src_v4 import run_pipeline

results = run_pipeline(
    document_path      = DOCUMENT_PATH,
    yaml_path          = YAML_PATH,
    question_ids       = QUESTION_IDS,
    verbose            = True,
    force_replan       = FORCE_REPLAN,
    use_chunk_cache    = USE_CACHE,
    concurrency        = CONCURRENCY,
    load_store         = LOAD_STORE,
)

In [ ]:
# ── Review answers ────────────────────────────────────────────────────────────
for r in results:
    print(f"[{r['question_id']}] ({r['question_type']})")
    print(f"  Q: {r['question'][:100]}")
    print(f"  A: {r['answer'][:200]}")
    print()

In [ ]:
# ── Inspect KnowledgeStore ────────────────────────────────────────────────────
from src_v4.knowledge_store import KnowledgeStore

store = KnowledgeStore.load('outputs_v4/knowledge_store.json')
print('Stats:', store.stats())

# Example queries
print('\n--- National parks ---')
for e in store.entities_by_type('national_park')[:5]:
    print(f"  {e['name']}  attrs={list(e.get('attributes',{}).keys())}")

print('\n--- Area facts (top 5) ---')
area_facts = [f for f in store.facts_by_predicate('area') if f.get('normalized_value')]
area_facts.sort(key=lambda x: x['normalized_value'], reverse=True)
for f in area_facts[:5]:
    print(f"  {f['subject']}: {f['value']} {f['unit']}  (norm={f['normalized_value']:.1f} km²)")

In [ ]:
# ── Inspect demand plan ───────────────────────────────────────────────────────
import json

with open('outputs_v4/demand_plan.json') as f:
    plan = json.load(f)

print('entity_types:',       plan.get('entity_types', []))
print('numeric_predicates:', plan.get('numeric_predicates', []))
print('theme_topics:',       plan.get('theme_topics', []))
print()
print('Question plans:')
for qid, qp in plan.get('question_plans', {}).items():
    print(f'  {qid}: {qp}')

In [ ]:
# store 객체는 pipeline 실행 후 접근 가능해야 함
keywords = ["poach", "Natura 2000", "accessibility", "brown bear", "wheelchair", "disabled"]

for kw in keywords:
    found_in = []
    for i, chunk in enumerate(store._raw_chunks):
        text = chunk.get("text", "")
        if kw.lower() in text.lower():
            found_in.append(i)
    print(f"'{kw}': chunks {found_in if found_in else '→ NOT FOUND in any chunk'}")

In [ ]:
import json

with open("outputs_v4/knowledge_store.json", encoding="utf-8") as f:
    ks = json.load(f)

targets = ["curonian", "wadden", "lithuania", "kaliningrad", "denmark", "netherlands"]

found = False
for section, records in ks.items():
    if not isinstance(records, list):
        continue
    for r in records:
        if any(t in str(r).lower() for t in targets):
            print(f"[{section}]", r)
            found = True

if not found:
    print("없음 → Phase 2에서 못 잡은 것")